# DEP Workbench — Interactive Analyst Notebook

Welcome to your **DEP-integrated JupyterLite** analytical playground. This environment runs a full Python kernel inside WebAssembly directly in your web browser. No remote servers are needed, and all computations happen securely on your machine.

This notebook demonstrates how to load authorized datasets, run statistics, generate charts aligned with the workbench design system, and save report outputs.

## 1. Establishing Session Context
First, we import the `dep_sdk` helper. The SDK detects your browser session token and automatically restricts catalog access based on your workbench role limits.

In [ ]:
import dep_sdk as dep

# Check active credentials, user roles, and allowed connections
dep.whoami()

## 2. Listing & Loading Allowed Data Catalogs
Let's see what catalog databases have been allocated to you by Governance policies, and load them as standard Pandas DataFrames.

In [ ]:
# List accessible catalogs
dep.list_catalogs()

In [ ]:
# Load data catalogs
df_customers = dep.get_catalog("customer_profiles")
df_revenue = dep.get_catalog("revenue_forecasting_db")
df_inventory = dep.get_catalog("product_inventory")

print("✅ All active catalogs loaded successfully!")
print(f"Loaded customer profiles shape: {df_customers.shape}")
display(df_customers.head())

## 3. Loading a CSV Dataset via DEP SDK

The `sales_transactions` catalog is backed by a real **CSV file** (`sales_transactions.csv`) that lives in your JupyterLite workspace. The DEP SDK resolves it through three layers:

1. **Filesystem lookup** — reads `/files/sales_transactions.csv` from the Pyodide virtual drive  
2. **Static server fetch** — XHR from `http://localhost:3000/jupyterlite/files/sales_transactions.csv`  
3. **Inline fallback** — an embedded 5-row sample when offline

You can also load **any CSV** you upload to the workspace using `dep.read_csv('filename.csv')`.

In [ ]:
# ── Method 1: Load via governed catalog name ───────────────────────────────────
df_sales = dep.get_catalog("sales_transactions")

print(f"Shape: {df_sales.shape}")
print(f"Columns: {list(df_sales.columns)}")
display(df_sales.head(10))

In [ ]:
# ── Method 2: Load directly by filename (for any CSV you upload) ──────────────
# dep.read_csv() wraps pd.read_csv with DEP governance logging
df_raw = dep.read_csv("sales_transactions.csv")

print(f"Loaded {len(df_raw)} transactions")
display(df_raw.dtypes.rename('dtype').to_frame())

## 4. Analysing the Sales Transactions Dataset

Now let's run typical analyst workflows — segment revenue, find top products, and compute profit margins.

In [ ]:
import pandas as pd

# Parse date column
df_sales['date'] = pd.to_datetime(df_sales['date'])
df_sales['month'] = df_sales['date'].dt.to_period('M').astype(str)

print("=== Dataset Overview ===")
print(f"Date range : {df_sales['date'].min().date()} → {df_sales['date'].max().date()}")
print(f"Transactions: {len(df_sales)}")
print(f"Customers   : {df_sales['customer_id'].nunique()} unique")
print(f"Regions     : {', '.join(df_sales['region'].unique())}")
print()

# Revenue by region
region_summary = df_sales.groupby('region').agg(
    transactions=('transaction_id', 'count'),
    total_revenue=('revenue', 'sum'),
    total_profit=('profit', 'sum'),
    avg_margin_pct=('profit', lambda x: (x.sum() / df_sales.loc[x.index, 'revenue'].sum() * 100).round(1))
).sort_values('total_revenue', ascending=False).round(2)

print("=== Revenue by Region ===")
display(region_summary)

In [ ]:
# Top 10 products by revenue
top_products = (
    df_sales.groupby('product_name')
    .agg(total_revenue=('revenue','sum'), total_units=('quantity','sum'), total_profit=('profit','sum'))
    .sort_values('total_revenue', ascending=False)
    .head(10)
    .round(2)
)
top_products['margin_%'] = (top_products['total_profit'] / top_products['total_revenue'] * 100).round(1)
print("=== Top 10 Products by Revenue ===")
display(top_products)

## 5. Visualisations — Sales Transactions Dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#141414')
fig.suptitle('Sales Transactions — DEP Analytics Dashboard', color='#e2e2e2', fontsize=15, y=1.01)

ACCENT  = ['#a78bfa','#6a9955','#569cd6','#ce9178','#e5c07b','#61afef']

for ax in axes.flat:
    ax.set_facecolor('#1e1e1e')
    ax.tick_params(colors='#a3a3a3', labelsize=8.5)
    ax.spines[:].set_color('#2b2b2b')
    ax.grid(axis='y', linestyle='--', alpha=0.15, color='#606060')

# ── Chart 1: Revenue by Region ─────────────────────────────────────────────────
ax = axes[0, 0]
bars = ax.bar(region_summary.index, region_summary['total_revenue'], color=ACCENT[:len(region_summary)])
ax.set_title('Total Revenue by Region', color='#cccccc', fontsize=11, pad=10)
ax.set_ylabel('Revenue (USD)', color='#a3a3a3')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticklabels(region_summary.index, rotation=15, ha='right')
for bar in bars:
    ax.annotate(f'${bar.get_height():,.0f}', (bar.get_x()+bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', color='#cccccc', fontsize=7.5, xytext=(0,3), textcoords='offset points')

# ── Chart 2: Revenue by Category ──────────────────────────────────────────────
ax = axes[0, 1]
cat_rev = df_sales.groupby('product_category')['revenue'].sum().sort_values(ascending=False)
wedges, texts, autotexts = ax.pie(
    cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%',
    colors=ACCENT[:len(cat_rev)], startangle=140,
    textprops={'color':'#cccccc','fontsize':8.5},
    wedgeprops={'edgecolor':'#141414','linewidth':1.5}
)
for at in autotexts: at.set_color('#e2e2e2')
ax.set_facecolor('#1e1e1e')
ax.set_title('Revenue Share by Product Category', color='#cccccc', fontsize=11, pad=10)

# ── Chart 3: Monthly Revenue Trend ────────────────────────────────────────────
ax = axes[1, 0]
monthly = df_sales.groupby('month').agg(revenue=('revenue','sum'), profit=('profit','sum'))
ax.plot(monthly.index, monthly['revenue'], marker='o', color='#569cd6', linewidth=2.2, label='Revenue')
ax.plot(monthly.index, monthly['profit'],  marker='s', color='#6a9955', linewidth=2.2, label='Profit', linestyle='--')
ax.fill_between(monthly.index, monthly['profit'], alpha=0.12, color='#6a9955')
ax.set_title('Monthly Revenue & Profit Trend', color='#cccccc', fontsize=11, pad=10)
ax.set_ylabel('Amount (USD)', color='#a3a3a3')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticklabels(monthly.index, rotation=20, ha='right')
ax.legend(facecolor='#141414', edgecolor='#2b2b2b', labelcolor='#cccccc', fontsize=8.5)

# ── Chart 4: Channel Distribution ─────────────────────────────────────────────
ax = axes[1, 1]
channel = df_sales.groupby('channel')['revenue'].sum()
ax.barh(channel.index, channel.values, color=ACCENT[:len(channel)])
ax.set_title('Revenue by Sales Channel', color='#cccccc', fontsize=11, pad=10)
ax.set_xlabel('Revenue (USD)', color='#a3a3a3')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.spines[:].set_color('#2b2b2b')
ax.grid(axis='x', linestyle='--', alpha=0.15, color='#606060')
ax.grid(axis='y', linestyle='none')

plt.tight_layout()
plt.show()

## 6. Data Wrangling & Analytical Summaries
We can perform complex aggregates on our pilot datasets using standard Pandas operators to summarize customer segment attributes.

In [ ]:
# Calculate recency, frequency and transaction summaries per customer segment
segment_summary = df_customers.groupby('segment').agg(
    average_recency_days=('recency_days', 'mean'),
    average_frequency_orders=('frequency', 'mean'),
    average_value_usd=('transaction_value', 'mean'),
    total_customers=('customer_id', 'count')
).round(2)

print("=== Customer Segment Analytics ===")
display(segment_summary)

## 7. Visualizations (Matplotlib & Seaborn Dark Theme)
Let's plot our pilot statistics. In order to match the **DEP Workbench Dark Theme**, we customize colors, border spines, and grid opacity to fit card backgrounds nicely.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set matplotlib to dark background template
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#141414') # Dark card background

# Apply customized styling rules for both axes subplots
for ax in axes:
    ax.set_facecolor('#1e1e1e') # Darker editor area background
    ax.tick_params(colors='#a3a3a3', labelsize=9)
    ax.spines[:].set_color('#2b2b2b') # Matching grey borders
    ax.grid(axis='y', linestyle='--', alpha=0.15, color='#606060')

# Left Subplot: Customer Segment Value Comparison
palette_colors = ['#a78bfa', '#6a9955', '#569cd6', '#ce9178', '#606060']
sns.barplot(x=segment_summary.index, y=segment_summary['average_value_usd'], 
            ax=axes[0], palette=palette_colors, hue=segment_summary.index, legend=False)
axes[0].set_title('Avg Transaction Value per Customer Segment', color='#cccccc', fontsize=12, pad=12)
axes[0].set_ylabel('Transaction Value (USD $)', color='#a3a3a3')
axes[0].set_xlabel('Cohort Segment', color='#a3a3a3')

# Add values over bars
for bar in axes[0].patches:
    axes[0].annotate(f"${bar.get_height():.2f}",
                   (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                   ha='center', va='bottom', color='#cccccc', fontsize=8.5, xytext=(0, 3),
                   textcoords='offset points')

# Right Subplot: actual revenue monthly trend and forecasts
axes[1].plot(df_revenue['month'], df_revenue['revenue'], marker='o', color='#6a9955', label='Actual Revenue', linewidth=2.5)
axes[1].plot(df_revenue['month'], df_revenue['forecast'], marker='s', linestyle='--', color='#ce9178', label='Forecast Projection', linewidth=2)
axes[1].fill_between(df_revenue['month'], df_revenue['confidence_lower'], df_revenue['confidence_upper'], 
                     color='#ce9178', alpha=0.08, label='95% Confidence Band')

axes[1].set_title('Monthly Revenue Performance vs. Forecasting Models', color='#cccccc', fontsize=12, pad=12)
axes[1].set_ylabel('Total Revenue (USD $)', color='#a3a3a3')
axes[1].set_xlabel('Reporting Month', color='#a3a3a3')
axes[1].legend(facecolor='#141414', edgecolor='#2b2b2b', labelcolor='#cccccc', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Saving, Storing & Exporting Outputs
Since JupyterLite runs locally in your browser sandbox, files are saved to the browser's persistent IndexedDB storage area. Let's see how to write dataframes and plots to files, which will appear immediately in the file manager sidebar.

In [ ]:
# 1. Export analytical tables as CSV format
segment_summary.to_csv("segment_analytics_report.csv", index=True)

# 2. Export detailed inventories as JSON documents
df_inventory.to_json("product_inventory_snapshot.json", orient="records", indent=2)

# 3. Export sales analysis
region_summary.to_csv("sales_region_analysis.csv", index=True)

print("🎉 SUCCESS!")
print("--------------------------------------------------")
print("Files saved to your workspace:")
print("  1. segment_analytics_report.csv  -> Customer Segment CSV")
print("  2. product_inventory_snapshot.json -> JSON Dataset")
print("  3. sales_region_analysis.csv      -> Sales Region CSV")
print("--------------------------------------------------")
print("👉 Check the left File Browser sidebar to view them. Right-click any file and select 'Download' to transfer it to your computer.")